# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

!pip install numpy pandas scipy scikit-learn sklearn tqdm cudf cupy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats

## Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 2.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cpu


# Functions

In [20]:
from google.colab import drive
import dill as pickle
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats


def trim_citations(citations, patent_ids):
    citations['patent_id'] = citations['patent_id'].astype(str)
    citations = citations[citations['patent_id'].isin(patent_ids)]
    print("Trimmed citations shape:", citations.shape)
    return citations


def precompute_quarterly_citations(citations):
    citations['citation_date'] = pd.to_datetime(citations['citation_date'])
    citations['year'] = citations['citation_date'].dt.year
    citations['month'] = citations['citation_date'].dt.month
    citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
    citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']
    grouped = citations.groupby(['patent_id', 'citation_quarter']).size()
    quarterly_counts_pd = grouped.unstack(fill_value=0)
    citation_counts_dict = {pid: row.to_dict() for pid, row in quarterly_counts_pd.iterrows()}
    return citation_counts_dict


def get_unique_ids(df):

    df['treated_id'] = df['treated_id'].astype(str)
    df['control_id'] = df['control_id'].astype(str)

    # Get unique IDs in a list
    patent_ids = list(set(df['treated_id'].unique()).union(set(df['control_id'].unique())))
    return patent_ids

def combine_with_citations(df, citation_counts_dict, periods_after=20):
    treated_cits, control_cits = [], []

    for _, row in df.iterrows():
        pre_q_list = row['pre_quarters']
        treatment_period = pd.Period(pre_q_list[-1], freq='Q') + 1

        # Dynamically determine number of pre-treatment quarters
        n_pre_periods = len(pre_q_list)

        # Build list of all quarters around treatment
        quarters = ([str(treatment_period - i) for i in range(n_pre_periods, 0, -1)] +
                    [str(treatment_period)] +
                    [str(treatment_period + i) for i in range(1, periods_after + 1)])

        # Label quarters
        quarter_labels = [f"q_{-n_pre_periods + i}" for i in range(n_pre_periods)] + \
                         [f"q_{i}" for i in range(0, periods_after + 1)]

        treated_id, control_id = str(row['treated_id']), str(row['control_id'])
        t_vec, c_vec = [], []

        for q in quarters:
            q_period = pd.Period(q, freq='Q')
            last_quarter = pd.Period("2024Q4", freq='Q')
            t_val = citation_counts_dict.get(treated_id, {}).get(q, 0)
            c_val = citation_counts_dict.get(control_id, {}).get(q, 0)
            t_vec.append(np.log1p(t_val) if q_period <= last_quarter else np.nan)
            c_vec.append(np.log1p(c_val) if q_period <= last_quarter else np.nan)

        treated_cits.append(t_vec)
        control_cits.append(c_vec)

    # Apply all quarter columns to DataFrame
    for i, label in enumerate(quarter_labels):
        df[label + "_treated"] = [vec[i] for vec in treated_cits]
        df[label + "_control"] = [vec[i] for vec in control_cits]

    df['match_id'] = df.index
    return df

def get_long_data(df, acq_type=None, years=None):
    df = df.copy()
    if acq_type:
        df['acq_type'] = acq_type
    if years:
        df['years'] = years

    id_cols = ['treated_id', 'control_id', 'match_id', 'mahalanobis_distance', 'cosine_distance']
    treated_cols = [col for col in df.columns if col.endswith("_treated") and col.startswith("q_")]
    control_cols = [col for col in df.columns if col.endswith("_control") and col.startswith("q_")]

    df_treated_long = df.melt(id_vars=id_cols, value_vars=treated_cols,
                              var_name="quarter_treated", value_name="log_citations_treated")
    df_control_long = df.melt(id_vars=id_cols, value_vars=control_cols,
                              var_name="quarter_control", value_name="log_citations_control")

    df_treated_long['quarter'] = df_treated_long['quarter_treated'].str.replace("_treated", "")
    df_control_long['quarter'] = df_control_long['quarter_control'].str.replace("_control", "")

    df_long = pd.merge(
        df_treated_long[id_cols + ['quarter', 'log_citations_treated']],
        df_control_long[id_cols + ['quarter', 'log_citations_control']],
        on=id_cols + ['quarter']
    )

    if acq_type:
        df_long['acq_type'] = acq_type
    if years:
        df_long['years'] = years

    df_long.sort_values(by=['treated_id', 'quarter'], inplace=True)
    return df_long


def process_sample(filepath, citations):
    sample = load(filepath)

    # Optional: filter matches
    sample = sample[sample['mahalanobis_distance'] <= 4]

    patent_ids = get_unique_ids(sample)
    trimmed_citations = trim_citations(citations, patent_ids)
    citation_counts_dict = precompute_quarterly_citations(trimmed_citations)
    enriched_sample = combine_with_citations(sample, citation_counts_dict)

    return enriched_sample


# Load citations & Consolidated patents info for acquired patents

In [6]:

# === Load citations ===
citations = pd.read_pickle("/content/drive/MyDrive/PhD Data/08 Citations/03 Patent citations - raw, filing.pickle")
citations.rename(columns={'patent_id': 'citedby_patent_id', 'citation_patent_id':'patent_id', 'filing_date':'citation_date'}, inplace=True)

# === Load patent metadata ===
all_patents_df = pd.read_stata("/content/drive/MyDrive/PhD Data/09 Acquired patents/04 All patents.dta")
all_patents_df['patent_id'] = all_patents_df['patent_id'].astype(str)


# Preparate and save

In [21]:
# === Define filepaths for 1y, 2y, 3y ===
filepaths_by_group = {
    ('M&A', 1): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_M&A_1y.pkl",
    ('M&A', 2): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_M&A_2y.pkl",
    ('M&A', 3): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_M&A_3y.pkl",
    ('Off deal', 1): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_Off_deal_1y.pkl",
    ('Off deal', 2): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_Off_deal_2y.pkl",
    ('Off deal', 3): "/content/drive/MyDrive/PhD Data/11 Matches/hybrid_Off_deal_3y.pkl",
}
# === Process and save all groups ===
for (acq_type, years), filepath in filepaths_by_group.items():
    print(f"Processing {acq_type}, {years}y")

    # Process match file and compute citations
    processed_df = process_sample(filepath, citations)

    # Convert to long format
    long_df = get_long_data(processed_df, acq_type=acq_type, years=years)

    # Drop NaNs and merge with patent metadata
    long_df = long_df.dropna(subset=['log_citations_treated', 'log_citations_control'])
    long_df = pd.merge(long_df, all_patents_df, left_on='treated_id', right_on='patent_id', how='inner')
    long_df['quarter'] = long_df['quarter'].apply(lambda x: x.replace('q_', '')).astype(int)

    # Save
    output_path = f"/content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, {acq_type}, {years}y, m4, filing date.csv"
    long_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")


Processing M&A, 1y
Trimmed citations shape: (172059, 4)


<ipython-input-20-91a806e08502>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-20-91a806e08502>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-20-91a806e08502>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

Saved: /content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, M&A, 1y, m4, filing date.csv
Processing M&A, 2y
Trimmed citations shape: (92185, 4)


<ipython-input-20-91a806e08502>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-20-91a806e08502>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-20-91a806e08502>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

Saved: /content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, M&A, 2y, m4, filing date.csv
Processing M&A, 3y
Trimmed citations shape: (40715, 4)


...

Saved: /content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, Off deal, 1y, m4, filing date.csv
Processing Off deal, 2y
Trimmed citations shape: (85394, 4)


<ipython-input-20-91a806e08502>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-20-91a806e08502>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-20-91a806e08502>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

Saved: /content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, Off deal, 2y, m4, filing date.csv
Processing Off deal, 3y
Trimmed citations shape: (34156, 4)


<ipython-input-20-91a806e08502>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['citation_date'] = pd.to_datetime(citations['citation_date'])
<ipython-input-20-91a806e08502>:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  citations['year'] = citations['citation_date'].dt.year
<ipython-input-20-91a806e08502>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

Saved: /content/drive/MyDrive/PhD Data/12 Sample Final/10 Sample - top tech, Off deal, 3y, m4, filing date.csv
